In [0]:
from pyspark.sql import functions as f
from pyspark.sql.types import *

data = [(1, '2017-01-01', 10), (2, '2017-01-02', 109), (3, '2017-01-03', 150), (4, '2017-01-04', 99), (5, '2017-01-05', 145), (6, '2017-01-06', 1455), (7, '2017-01-07', 199), (8, '2017-01-09', 188)]

schema = StructType(
    [
        StructField('id',IntegerType()),
        StructField('visit_date',StringType()),
        StructField('people',IntegerType()),
    ]
)

stadium = spark.createDataFrame(data=data,schema=schema)

starts = stadium.alias('s1').join(stadium.alias('s2'),on=f.col('s2.id')==f.col('s1.id')+1,how='inner').join(stadium.alias('s3'),on=f.col('s3.id')==f.col('s1.id')+2,how='inner').filter((f.col('s1.people')>=100) & (f.col('s2.people')>=100) & (f.col('s3.people')>=100)).select(f.col('s1.id'))

ids = starts.select(f.collect_list(f.col('id'))).first()[0]
ids.append(ids[0]+2)
ids.append(ids[0]+3)

res = stadium.alias('s').join(starts.alias('st'),on=f.col('s.id').isin(ids)).orderBy(f.col('s.id')).select('s.*').distinct()

res.show()